# Fabric Copilot Question Mining



Retrieve and sanitize questions asked to Fabric Data Agent and Power BI Copilot so authorized developers can build response-quality tests.



## Before you run



- Enable Microsoft Purview Audit.

- Enable **DSPM for AI - Capture interactions for Copilot experiences**.

- Enable the Fabric tenant setting **Allow Microsoft Purview to secure AI interactions**.

- Grant an Entra application the Office 365 Management APIs application permission `ActivityFeed.Read`, then grant tenant admin consent.

- Store the application secret in Azure Key Vault. Never paste it into this notebook.

- Attach a default Lakehouse only if Delta export is enabled.



> Fabric Data Agent audit logging is currently preview. Data Agent `CopilotInteraction` records can contain full prompts and responses. Other Copilot audit records can expose only message IDs through the Management Activity API. This notebook measures text coverage and does not infer missing prompts. Treat prompts and responses as sensitive tenant data and follow your organization's purpose, access, retention, and review policies.

In [ ]:
# 1. Install and import dependencies

# Fabric Runtime includes requests, pandas, matplotlib, and PySpark.

# If your Environment removes one, add it to the Environment artifact rather than

# installing packages on every scheduled run.



import hashlib

import json

import re

import time

from collections import Counter

from datetime import datetime, timedelta, timezone



import matplotlib.pyplot as plt

import pandas as pd

import requests



print("Dependencies loaded.")

In [ ]:
# 2. Configure Fabric tenant and log sources

# This cell can be overridden by a Fabric pipeline.



TENANT_ID = "00000000-0000-0000-0000-000000000000"

CLIENT_ID = "00000000-0000-0000-0000-000000000000"

KEY_VAULT_URL = "https://your-vault.vault.azure.net/"

CLIENT_SECRET_NAME = "fabric-audit-reader-client-secret"



DAYS_BACK = 30

CONTENT_TYPES = ["Audit.General"]

START_MISSING_SUBSCRIPTIONS = False

HASH_USER_IDS = True

INCLUDE_RESPONSES = True

MAX_DISPLAY_ROWS = 100



SAVE_RAW_RESPONSES = False

RAW_AUDIT_PATH = "Files/restricted/fabric_copilot_audit_raw"

SAVE_TO_DELTA = False

OUTPUT_TABLE = "fabric_copilot_question_queue"

RETENTION_DAYS = 90



SEARCH_TEXT = ""

FILTER_PRODUCT = ""

FILTER_WORKSPACE = ""

FILTER_START_UTC = None

FILTER_END_UTC = None



if not 1 <= DAYS_BACK <= 7:

    raise ValueError("DAYS_BACK must be between 1 and 7 for this Activity API workflow.")

if TENANT_ID.startswith("00000000") or CLIENT_ID.startswith("00000000"):

    raise ValueError("Set TENANT_ID and CLIENT_ID before running the notebook.")

if not KEY_VAULT_URL.startswith("https://"):

    raise ValueError("KEY_VAULT_URL must be an HTTPS Azure Key Vault URL.")



print(f"Configured a {DAYS_BACK}-day UTC lookback for {CONTENT_TYPES}.")

In [ ]:
# 3. Authenticate with Microsoft Entra ID



client_secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, CLIENT_SECRET_NAME)

token_response = requests.post(

    f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",

    data={

        "client_id": CLIENT_ID,

        "client_secret": client_secret,

        "grant_type": "client_credentials",

        "scope": "https://manage.office.com/.default",

    },

    timeout=60,

)

token_response.raise_for_status()

access_token = token_response.json()["access_token"]

session = requests.Session()

session.headers.update({"Authorization": f"Bearer {access_token}"})

api_base = f"https://manage.office.com/api/v1.0/{TENANT_ID}/activity/feed"



def api_request(method, url, *, params=None, max_attempts=6):

    if max_attempts < 1:

        raise ValueError("max_attempts must be at least 1.")

    request_params = dict(params or {})

    request_params["PublisherIdentifier"] = TENANT_ID

    for attempt in range(max_attempts):

        response = session.request(method, url, params=request_params, timeout=120)

        if response.status_code not in {429, 500, 502, 503, 504}:

            if not response.ok:

                raise RuntimeError(

                    f"Activity API {method} failed ({response.status_code}): "

                    f"{response.text[:1000]}"

                )

            return response

        if attempt == max_attempts - 1:

            response.raise_for_status()

        retry_after = int(response.headers.get("Retry-After", 2 ** attempt))

        time.sleep(min(retry_after, 60))

    raise RuntimeError("Activity API retry loop ended unexpectedly.")



validation = api_request("GET", f"{api_base}/subscriptions/list")

print(f"Authorization validated. Found {len(validation.json())} subscription(s).")

In [ ]:
# 4. Check or start audit content subscriptions



subscriptions = validation.json()

enabled_types = {

    item.get("contentType")

    for item in subscriptions

    if item.get("status", "enabled").lower() == "enabled"

}

missing_types = [item for item in CONTENT_TYPES if item not in enabled_types]



if missing_types and START_MISSING_SUBSCRIPTIONS:

    for content_type in missing_types:

        api_request(

            "POST",

            f"{api_base}/subscriptions/start",

            params={"contentType": content_type},

        )

        print(f"Started {content_type}. Initial content can take up to 12 hours.")

elif missing_types:

    raise RuntimeError(

        f"Missing subscription(s): {missing_types}. Set "

        "START_MISSING_SUBSCRIPTIONS=True if this app is approved to start them. "

        "Initial content can take up to 12 hours after a subscription starts."

    )

else:

    print(f"Required subscription(s) already enabled: {CONTENT_TYPES}")

In [ ]:
# 5. Retrieve Fabric and Power BI activity logs



def utc_api_time(value):

    return value.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")



end_utc = datetime.now(timezone.utc).replace(microsecond=0)

start_utc = end_utc - timedelta(days=DAYS_BACK)

content_by_uri = {}



window_start = start_utc

while window_start < end_utc:

    window_end = min(window_start + timedelta(hours=24), end_utc)

    for content_type in CONTENT_TYPES:

        next_url = f"{api_base}/subscriptions/content"

        next_params = {

            "contentType": content_type,

            "startTime": utc_api_time(window_start),

            "endTime": utc_api_time(window_end),

        }

        while next_url:

            response = api_request("GET", next_url, params=next_params)

            for item in response.json():

                content_by_uri[item["contentUri"]] = item

            next_url = response.headers.get("NextPageUri")

            next_params = None

    window_start = window_end



raw_records = []

for index, item in enumerate(content_by_uri.values(), start=1):

    response = api_request("GET", item["contentUri"])

    payload = response.json()

    raw_records.extend(payload if isinstance(payload, list) else [payload])

    if index % 100 == 0:

        print(f"Downloaded {index:,} of {len(content_by_uri):,} content blobs.")



if SAVE_RAW_RESPONSES and raw_records:

    raw_json_df = spark.createDataFrame(

        [(json.dumps(record, separators=(",", ":")),) for record in raw_records],

        ["value"],

    )

    raw_json_df.write.mode("overwrite").text(RAW_AUDIT_PATH)

    print(f"Raw records saved to restricted path: {RAW_AUDIT_PATH}")



print(f"Retrieved {len(raw_records):,} records from {len(content_by_uri):,} blobs.")

In [ ]:
# 6. Load telemetry and discover the tenant schema without exposing values



def decode_nested_json(value):

    if not isinstance(value, str):

        return value

    stripped = value.strip()

    if not stripped or stripped[0] not in "[{":

        return value

    try:

        return json.loads(stripped)

    except json.JSONDecodeError:

        return value



def walk_nodes(value, path=""):

    value = decode_nested_json(value)

    if isinstance(value, dict):

        yield path, value

        for key, child in value.items():

            child_path = f"{path}.{key}" if path else key

            yield from walk_nodes(child, child_path)

    elif isinstance(value, list):

        for child in value:

            yield from walk_nodes(child, f"{path}[]")



def ci_value(mapping, *names):

    if not isinstance(mapping, dict):

        return None

    lowered = {str(key).casefold(): value for key, value in mapping.items()}

    for name in names:

        if name.casefold() in lowered:

            return decode_nested_json(lowered[name.casefold()])

    return None



def recursive_first(record, *names):

    for _, node in walk_nodes(record):

        value = ci_value(node, *names)

        if value not in (None, "", [], {}):

            return value

    return None



profile_rows = []

sensitive_path_terms = ("prompt", "question", "message", "content", "text", "response", "answer")

candidate_paths = Counter()

for record in raw_records:

    profile_rows.append({

        "Operation": record.get("Operation"),

        "Workload": record.get("Workload"),

        "AppIdentity": recursive_first(record, "AppIdentity", "AppName"),

        "AppHost": recursive_first(record, "AppHost"),

    })

    for path, _ in walk_nodes(record):

        if any(term in path.casefold() for term in sensitive_path_terms):

            candidate_paths[path] += 1



profile_df = pd.DataFrame(profile_rows)

print(f"Total records: {len(profile_df):,}")

if not profile_df.empty:

    display(

        profile_df.value_counts(dropna=False)

        .rename("event_count")

        .reset_index()

        .head(30)

    )

display(pd.DataFrame(candidate_paths.most_common(100), columns=["property_path", "record_count"]))

In [ ]:
# 7. Extract and normalize Data Agent and Power BI Copilot interactions



def scalar_text(value):

    value = decode_nested_json(value)

    if not isinstance(value, str):

        return None

    text = " ".join(value.split()).strip()

    if not text or re.fullmatch(r"[0-9a-fA-F-]{20,}", text):

        return None

    return text



def identity_text(value):

    if isinstance(value, dict):

        value = ci_value(value, "Name", "Id", "Value")

    return scalar_text(value) or ""



def event_signature(record):

    fields = [

        record.get("Operation"),

        record.get("Workload"),

        recursive_first(record, "AppIdentity", "AppName"),

        recursive_first(record, "AppHost"),

        recursive_first(record, "AgentName"),

    ]

    return " ".join(identity_text(value) for value in fields).casefold()



def is_fabric_ai_event(record):

    operation = str(record.get("Operation", ""))

    normalized = operation.casefold().replace(" ", "")

    if operation == "FabricCopilotSessionMessageSent":

        return True

    if "fabriccopilot" in normalized or "dataagent" in normalized:

        return True

    if operation == "CopilotInteraction":

        signature = event_signature(record)

        return any(term in signature for term in ("fabric", "powerbi", "power bi"))

    return False



def classify_product(record):

    signature = event_signature(record)

    if "fabric-data agent" in signature or "fabric data agent" in signature or "dataagent" in signature:

        return "Fabric Data Agent"

    if "powerbi" in signature or "power bi" in signature:

        return "Power BI Copilot"

    return "Fabric Copilot"



def append_unique(items, value):

    text = scalar_text(value)

    if text and text.casefold() not in {item.casefold() for item in items}:

        items.append(text)



def extract_messages(record):

    prompts = []

    responses = []

    for _, node in walk_nodes(record):

        for key in ("Prompt", "UserPrompt", "Question"):

            append_unique(prompts, ci_value(node, key))

        for key in ("Response", "Answer", "GeneratedResponse"):

            append_unique(responses, ci_value(node, key))



        is_prompt = ci_value(node, "isPrompt")

        role = str(ci_value(node, "Role", "AuthorRole", "Type") or "").casefold()

        message_text = ci_value(node, "Content", "Text", "Message", "Body")

        if is_prompt is True or role in {"user", "human", "prompt"}:

            append_unique(prompts, message_text)

        elif is_prompt is False or role in {"assistant", "copilot", "agent", "response"}:

            append_unique(responses, message_text)

    return prompts, responses



def pseudonymize_user(value):

    text = str(value or "")

    if not text:

        return None

    if not HASH_USER_IDS:

        return text

    return hashlib.sha256(f"{TENANT_ID}:{text.casefold()}".encode("utf-8")).hexdigest()[:16]



extracted_rows = []

for record in raw_records:

    if not is_fabric_ai_event(record):

        continue

    prompts, responses = extract_messages(record)

    common = {

        "event_id": record.get("Id") or record.get("RecordId"),

        "timestamp_utc": record.get("CreationTime") or record.get("CreationDate"),

        "operation": record.get("Operation"),

        "product": classify_product(record),

        "workspace": identity_text(recursive_first(record, "WorkSpaceName", "WorkspaceName")),

        "app_identity": identity_text(recursive_first(record, "AppIdentity", "AppName")),

        "app_host": identity_text(recursive_first(record, "AppHost")),

        "agent_name": identity_text(recursive_first(record, "AgentName")),

        "thread_id": identity_text(recursive_first(record, "ThreadId")),

        "user_id": pseudonymize_user(record.get("UserId") or record.get("UserKey")),

    }

    if prompts:

        for position, prompt in enumerate(prompts):

            response = responses[position] if position < len(responses) else (responses[0] if responses else None)

            extracted_rows.append({

                **common,

                "question": prompt,

                "response": response if INCLUDE_RESPONSES else None,

                "has_question": True,

                "missing_text_reason": None,

            })

    else:

        extracted_rows.append({

            **common,

            "question": None,

            "response": responses[0] if responses and INCLUDE_RESPONSES else None,

            "has_question": False,

            "missing_text_reason": "Audit event contains metadata/message IDs but no prompt text",

        })



audit_events_df = pd.DataFrame(extracted_rows)

print(f"Normalized {len(audit_events_df):,} Fabric AI interaction row(s); text is not displayed yet.")

In [ ]:
# 8. Redact sensitive information before display or export



AUDIT_COLUMNS = [

    "event_id", "timestamp_utc", "operation", "product", "workspace",

    "app_identity", "app_host", "agent_name", "thread_id", "user_id",

    "question", "response", "has_question", "missing_text_reason",

]

audit_events_df = audit_events_df.reindex(columns=AUDIT_COLUMNS)



REDACTION_PATTERNS = [

    (re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.I), "[EMAIL]"),

    (re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}\b", re.I), "[GUID]"),

    (re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"), "[IP_ADDRESS]"),

    (re.compile(r"(?<!\w)(?:\+?\d[\d .()\-]{7,}\d)(?!\w)"), "[PHONE_OR_NUMBER]"),

    (re.compile(r"(?i)\b(api[_ -]?key|client[_ -]?secret|password|token)\s*[:=]\s*\S+"), r"\1=[REDACTED]"),

    (re.compile(r"\beyJ[A-Za-z0-9_-]{20,}\.[A-Za-z0-9_-]{10,}(?:\.[A-Za-z0-9_-]{10,})?\b"), "[TOKEN]"),

]



def redact_text(value):

    if value is None or pd.isna(value):

        return None

    text = str(value)

    for pattern, replacement in REDACTION_PATTERNS:

        text = pattern.sub(replacement, text)

    return text



for column in ["question", "response"]:

    audit_events_df[column] = audit_events_df[column].map(redact_text)



# Event and thread IDs are not needed in the developer-facing dataset.

audit_events_df = audit_events_df.drop(columns=["event_id", "thread_id"])

raw_records.clear()

del extracted_rows



print("Sensitive text patterns redacted; raw in-memory records cleared.")

In [ ]:
# 9. Search and filter sanitized questions



if audit_events_df.empty:

    coverage_df = pd.DataFrame(columns=["product", "events", "questions", "question_coverage_pct"])

else:

    coverage_df = (

        audit_events_df.groupby("product", dropna=False)

        .agg(events=("has_question", "size"), questions=("has_question", "sum"))

        .reset_index()

    )

    coverage_df["question_coverage_pct"] = (

        100 * coverage_df["questions"] / coverage_df["events"]

    ).round(1)

display(coverage_df)



question_rows = audit_events_df[audit_events_df["has_question"].fillna(False)].copy()

question_rows["timestamp_utc"] = pd.to_datetime(question_rows["timestamp_utc"], utc=True, errors="coerce")

question_rows["question_key"] = question_rows["question"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()



def first_nonempty(values):

    for value in values:

        if value is not None and not pd.isna(value) and str(value).strip():

            return value

    return None



if question_rows.empty:

    questions_df = pd.DataFrame(columns=[

        "product", "question", "response", "frequency", "first_seen_utc",

        "last_seen_utc", "workspace", "agent_name",

    ])

else:

    questions_df = (

        question_rows.groupby(["product", "question_key"], dropna=False)

        .agg(

            question=("question", "first"),

            response=("response", first_nonempty),

            frequency=("question", "size"),

            first_seen_utc=("timestamp_utc", "min"),

            last_seen_utc=("timestamp_utc", "max"),

            workspace=("workspace", first_nonempty),

            agent_name=("agent_name", first_nonempty),

        )

        .reset_index()

        .drop(columns=["question_key"])

        .sort_values(["frequency", "last_seen_utc"], ascending=[False, False])

    )



def as_utc_timestamp(value):

    timestamp = pd.Timestamp(value)

    return timestamp.tz_localize("UTC") if timestamp.tzinfo is None else timestamp.tz_convert("UTC")



filtered_questions_df = questions_df.copy()

if SEARCH_TEXT:

    filtered_questions_df = filtered_questions_df[

        filtered_questions_df["question"].str.contains(SEARCH_TEXT, case=False, regex=False, na=False)

    ]

if FILTER_PRODUCT:

    filtered_questions_df = filtered_questions_df[

        filtered_questions_df["product"].str.casefold() == FILTER_PRODUCT.casefold()

    ]

if FILTER_WORKSPACE:

    filtered_questions_df = filtered_questions_df[

        filtered_questions_df["workspace"].str.contains(FILTER_WORKSPACE, case=False, regex=False, na=False)

    ]

if FILTER_START_UTC:

    filtered_questions_df = filtered_questions_df[

        filtered_questions_df["last_seen_utc"] >= as_utc_timestamp(FILTER_START_UTC)

    ]

if FILTER_END_UTC:

    filtered_questions_df = filtered_questions_df[

        filtered_questions_df["first_seen_utc"] <= as_utc_timestamp(FILTER_END_UTC)

    ]



if not audit_events_df.empty and question_rows.empty:

    print(

        "WARNING: Relevant events were found, but this API exposed no prompt text. "

        "Verify the DSPM capture policy and Fabric tenant setting, then inspect or "

        "export Copilot Interaction records from Purview Activity Explorer."

    )



display(filtered_questions_df.head(MAX_DISPLAY_ROWS))

In [ ]:
# 10. Analyze question patterns



analysis_df = filtered_questions_df.copy()

if analysis_df.empty:

    print("No sanitized questions match the current filters.")

else:

    analysis_df["question_length"] = analysis_df["question"].str.len()

    stop_words = {

        "a", "an", "and", "are", "for", "from", "how", "in", "is", "me",

        "of", "on", "show", "the", "to", "what", "which", "with",

    }

    topic_counts = Counter(

        word.casefold()

        for question in analysis_df["question"]

        for word in re.findall(r"[A-Za-z][A-Za-z0-9_-]{2,}", question)

        if word.casefold() not in stop_words

    )



    summary = pd.DataFrame({

        "metric": ["unique questions", "total occurrences", "median length", "max frequency"],

        "value": [

            len(analysis_df),

            int(analysis_df["frequency"].sum()),

            float(analysis_df["question_length"].median()),

            int(analysis_df["frequency"].max()),

        ],

    })

    display(summary)

    display(pd.DataFrame(topic_counts.most_common(20), columns=["topic_term", "count"]))



    figure, axes = plt.subplots(1, 2, figsize=(12, 4))

    analysis_df.groupby("product")["frequency"].sum().sort_values().plot.barh(

        ax=axes[0], title="Question occurrences by product"

    )

    analysis_df["question_length"].plot.hist(

        bins=20, ax=axes[1], title="Sanitized question length"

    )

    axes[0].set_xlabel("Occurrences")

    axes[1].set_xlabel("Characters")

    plt.tight_layout()

    plt.show()



    trend_df = (

        analysis_df.assign(day=analysis_df["last_seen_utc"].dt.date)

        .groupby(["day", "product"])["frequency"]

        .sum()

        .unstack(fill_value=0)

    )

    if not trend_df.empty:

        trend_df.plot(figsize=(12, 4), marker="o", title="Question occurrences by day")

        plt.ylabel("Occurrences")

        plt.tight_layout()

        plt.show()

In [ ]:
# 11. Create a response-quality test dataset



evaluation_df = questions_df.copy()

if evaluation_df.empty:

    evaluation_df = pd.DataFrame(columns=[

        "test_case_id", "product", "question", "category", "priority",

        "source_event_count", "expected_behavior", "expected_answer",

        "observed_response", "actual_response", "accuracy_score",

        "grounding_score", "safety_pass", "latency_ms", "status",

        "reviewer_notes", "first_seen_utc", "last_seen_utc",

    ])

else:

    evaluation_df["test_case_id"] = evaluation_df.apply(

        lambda row: "FQ-" + hashlib.sha256(

            f"{row['product']}|{row['question'].casefold()}".encode("utf-8")

        ).hexdigest()[:12].upper(),

        axis=1,

    )

    evaluation_df["category"] = ""

    evaluation_df["priority"] = evaluation_df["frequency"].map(

        lambda count: "High" if count >= 10 else ("Medium" if count >= 3 else "Normal")

    )

    evaluation_df["source_event_count"] = evaluation_df["frequency"]

    evaluation_df["expected_behavior"] = ""

    evaluation_df["expected_answer"] = ""

    evaluation_df["observed_response"] = evaluation_df["response"]

    evaluation_df["actual_response"] = ""

    evaluation_df["accuracy_score"] = pd.NA

    evaluation_df["grounding_score"] = pd.NA

    evaluation_df["safety_pass"] = pd.NA

    evaluation_df["latency_ms"] = pd.NA

    evaluation_df["status"] = "New"

    evaluation_df["reviewer_notes"] = ""

    evaluation_df = evaluation_df[[

        "test_case_id", "product", "question", "category", "priority",

        "source_event_count", "expected_behavior", "expected_answer",

        "observed_response", "actual_response", "accuracy_score",

        "grounding_score", "safety_pass", "latency_ms", "status",

        "reviewer_notes", "first_seen_utc", "last_seen_utc",

    ]]



display(evaluation_df.head(MAX_DISPLAY_ROWS))

In [ ]:
# 12. Export sanitized results securely



extracted_at_utc = datetime.now(timezone.utc).replace(microsecond=0)

retention_expires_utc = extracted_at_utc + timedelta(days=RETENTION_DAYS)

audit_summary_df = coverage_df.copy()

audit_summary_df["lookback_start_utc"] = start_utc.isoformat()

audit_summary_df["lookback_end_utc"] = end_utc.isoformat()

audit_summary_df["extracted_at_utc"] = extracted_at_utc.isoformat()

audit_summary_df["retention_expires_utc"] = retention_expires_utc.isoformat()

audit_summary_df["raw_records_persisted"] = SAVE_RAW_RESPONSES



if SAVE_TO_DELTA:

    if evaluation_df.empty:

        raise RuntimeError("No sanitized questions are available to export.")

    export_df = evaluation_df.copy()

    export_df["extracted_at_utc"] = extracted_at_utc.isoformat()

    export_df["retention_expires_utc"] = retention_expires_utc.isoformat()

    for column in ["first_seen_utc", "last_seen_utc"]:

        export_df[column] = export_df[column].map(

            lambda value: value.isoformat() if pd.notna(value) else None

        )

    export_df = export_df.astype(object).where(pd.notna(export_df), None)



    (

        spark.createDataFrame(export_df)

        .write.format("delta")

        .mode("overwrite")

        .option("overwriteSchema", "true")

        .saveAsTable(OUTPUT_TABLE)

    )

    (

        spark.createDataFrame(audit_summary_df.astype(object).where(pd.notna(audit_summary_df), None))

        .write.format("delta")

        .mode("overwrite")

        .option("overwriteSchema", "true")

        .saveAsTable(f"{OUTPUT_TABLE}_audit_summary")

    )

    print(f"Saved sanitized evaluation queue to {OUTPUT_TABLE}.")

    print(f"Retention review date: {retention_expires_utc.isoformat()}")

else:

    print("SAVE_TO_DELTA is False; no developer-facing data was persisted.")

    display(audit_summary_df)